# Phase 6.7 — market-relative feature comparison

This notebook tests three features that compare each company with the wider US market: SPY momentum, the company’s performance above or below SPY, and beta. It first checks that these SPY-based values are present on every historical row used by the experiment.

The project predicts after each daily market session closes. Exact intraday data-arrival times are not available in the source, so this is a daily-session alignment check, not a minute-by-minute timing claim.

In [ ]:
from pathlib import Path
import subprocess
import sys

BRANCH = 'feature/model-evaluation'
COLAB_REPOSITORY = Path('/content/HalalifyAI')
if 'google.colab' in sys.modules:
    if not (COLAB_REPOSITORY / '.git').is_dir():
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/frager098/HalalifyAI.git', str(COLAB_REPOSITORY)], check=True)
    root = COLAB_REPOSITORY.resolve()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt'), '-r', str(root / 'requirements-cleaning.txt'), 'gdown>=5.2,<6'], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'src' / 'model_training').is_dir():
        root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.notebook_environment import ensure_phase5_dataset, prepare_repository
root = prepare_repository(branch=BRANCH, colab_repository=COLAB_REPOSITORY)
ensure_phase5_dataset(root)
%cd {root}
print(f'Project folder: {root}')

## Run the alignment check and controlled comparison

The next cell checks daily SPY alignment, then compares all 14 approved market inputs with the same set without the three market-relative inputs. It tunes only on earlier inner-validation data and scores each later outer fold once. Expect a few minutes of work.

In [ ]:
from src.model_training.feature_ablation.phase5_continuation import (
    MARKET_RELATIVE_OUTPUT,
    prepare_inputs,
    write_market_relative_report,
)

prepare_inputs()
report = write_market_relative_report()
print('Status:', report['status'])
print('SPY alignment:', report['market_relative_alignment']['status'])
print('Evidence folder:', MARKET_RELATIVE_OUTPUT)
for task, summary in report['task_summaries'].items():
    print(f"{task}: {summary['decision']} | {summary['primary_metric']} difference = {summary['mean_difference']:.4f}")

## Read the result

`retain` means the three SPY-related features consistently helped. `reject` means they did not help in most periods. `inconclusive` means the evidence is mixed. For risk, a result cannot be retained if High-risk detection becomes worse or more High-risk outcomes are wrongly predicted as Low risk.

In [ ]:
print('Alignment note:', report['market_relative_alignment']['source_timing_limitation'])
for task, summary in report['task_summaries'].items():
    print('\n', task.upper())
    display(summary['fold_metrics'])
    if task == 'risk':
        print('Risk safety checks:', summary['risk_guardrails'])